# Using pandas groupby Effectively

Companion to the [complete article](https://todatabeyond.com/blog/how-to-use-groupby-effectively-as-a-data-scientist). Run all cells in order in Python 3.11+ on CPU. No API keys or GPU. Setup installs pinned pandas 3.1.0, NumPy 2.3.3 and certifi; the original author CSV downloads over certificate-verified HTTPS at an immutable commit and is SHA-256 checked.

Retains all 18 source examples. Select numeric columns explicitly for z-scores, use label-based DataFrame writes instead of chained assignments, and preserve an unchanged original frame across demonstrations. The sex-based comparison standardizes total_bill in both methods. Use exact integer summation and perf_counter. Timings are single current elapsed samples, not universal rankings.

Seed the original probabilistic missing-value construction with default_rng(42): p=0.1 produces 21 missing bills in this 244-row file, not an exact 10% quota. Recount restaurant_nan rather than the original frame. Assign transform output to total_bill; the manual baseline fills only missing total_bill cells (not the erroneous total_time column and not every observed value). Replace removed Series.append with concat, retain complete selected groups, and verify pooled tips. Group-concatenated tips may have a different order/index from filter; compare their full values and weighted mean.

The original historical prose/screenshot counts and timing claims remain in the blog; current Output panels come from this execution. Group means use observed values only; all-missing groups remain missing in the edge test. z-scores use sample std (ddof=1). FutureWarning and chained-assignment warnings are errors during execution. No measured outputs/counts are committed.


In [ ]:
%pip install --quiet pandas==3.1.0 numpy==2.3.3 certifi==2025.10.5
from importlib.metadata import version
print("pandas:", version("pandas"), "NumPy:", version("numpy"))


In [ ]:
import hashlib
import io
import ssl
import warnings
from urllib.request import urlopen
import certifi
import numpy as np
import pandas as pd

RESTAURANT_URL = "https://raw.githubusercontent.com/youssefHosni/Efficient-Python-for-Data-Scientists-Book/2507c7f72e3cb08ef349047a4ca052bcc7307337/Datasets/restaurant_data.csv"
RESTAURANT_SHA256 = "e54cc4d2ce1bff65d32ca60b3e4b802e06bde1d7e7caf6f796f6bf7370e863b0"

def load_restaurant_data():
    context = ssl.create_default_context(cafile=certifi.where())
    with urlopen(RESTAURANT_URL, context=context, timeout=60) as response:
        payload = response.read()
    assert hashlib.sha256(payload).hexdigest() == RESTAURANT_SHA256
    frame = pd.read_csv(io.BytesIO(payload))
    print("restaurant_data.csv shape:", frame.shape)
    return frame

warnings.filterwarnings("error", category=FutureWarning)
warnings.filterwarnings("error", category=pd.errors.ChainedAssignmentError)
print("Restaurant loader ready; HTTPS and SHA-256 are verified.")


## Example 1


In [ ]:
import time
# record time before execution
start_time = time.perf_counter()
# execute operation
result = 5 + 2
# record time after execution
end_time = time.perf_counter()
print("Result calculated in {} sec".format(end_time - start_time))
print("Result:", result)


## Example 2


In [ ]:
#using List comprehension 

list_comp_start_time = time.perf_counter()
result = [i*i for i in range(0,1000000)]
list_comp_end_time = time.perf_counter()
print("Time using the list_comprehension: {} sec".format(list_comp_end_time -
list_comp_start_time))
squares_comprehension = result
print("Squares:", len(result), result[:3], result[-3:])


## Example 3


In [ ]:
# Using For loop

for_loop_start_time= time.perf_counter()
result=[]
for i in range(0,1000000):
  result.append(i*i)
for_loop_end_time= time.perf_counter()
print("Time using the for loop: {} sec".format(for_loop_end_time - for_loop_start_time))
squares_loop = result
print("Squares:", len(result), result[:3], result[-3:])


## Example 4


In [ ]:
list_comp_time = list_comp_end_time - list_comp_start_time
for_loop_time = for_loop_end_time - for_loop_start_time
print("Difference in time: {} %".format((for_loop_time - list_comp_time)/
list_comp_time*100))


## Example 5


In [ ]:
def sum_brute_force(N):
  res = 0
  for i in range(1,N+1):
    res+=i
  return res

# Using brute force
bf_start_time = time.perf_counter()
bf_result = sum_brute_force(1000000)
bf_end_time = time.perf_counter()

print("Time using brute force: {} sec".format(bf_end_time - bf_start_time))
print("Sum:", bf_result)


## Example 6


In [ ]:
def sum_formula(N):
  return N*(N+1)//2
  
# Using the formula
formula_start_time = time.perf_counter()
formula_result = sum_formula(1000000)
formula_end_time = time.perf_counter()

print("Time using the formula: {} sec".format(formula_end_time - formula_start_time))
print("Sum:", formula_result)


## Example 7


In [ ]:
restaurant = load_restaurant_data()
restaurant_original = restaurant.copy(deep=True)
restaurant_grouped = restaurant.groupby('smoker')
print(restaurant_grouped.count().to_string())


## Example 8


In [ ]:
zscore = lambda x: (x - x.mean()) / x.std()
numeric_columns = ['total_bill', 'tip', 'size']
restaurant_grouped = restaurant.groupby('time')[numeric_columns]
restaurant_transformed = restaurant_grouped.transform(zscore)
print(restaurant_transformed.head().to_string())


## Example 9


In [ ]:
start_transform = time.perf_counter()
sex_bill_zscore = restaurant.groupby('sex')['total_bill'].transform(zscore)
transform_time = time.perf_counter() - start_transform

manual_zscore = restaurant.copy(deep=True)
start_loop = time.perf_counter()
grouped_bills = restaurant.groupby('sex')['total_bill']
mean_female = grouped_bills.mean()['Female']
mean_male = grouped_bills.mean()['Male']
std_female = grouped_bills.std()['Female']
std_male = grouped_bills.std()['Male']
for index in restaurant.index:
    if restaurant.loc[index, 'sex'] == 'Female':
        manual_zscore.loc[index, 'total_bill'] = (restaurant.loc[index, 'total_bill'] - mean_female) / std_female
    else:
        manual_zscore.loc[index, 'total_bill'] = (restaurant.loc[index, 'total_bill'] - mean_male) / std_male
loop_time = time.perf_counter() - start_loop
print("Time using transform:", transform_time, "sec")
print("Time using the loop:", loop_time, "sec")
print(pd.DataFrame({'transform': sex_bill_zscore, 'loop': manual_zscore['total_bill']}).head().to_string())


## Example 10


In [ ]:
prior_counts = restaurant.groupby('time')
print(prior_counts['total_bill'].count().to_string())


## Example 11


In [ ]:
p = 0.1
rng = np.random.default_rng(42)
mask = rng.choice([np.nan, 1], size=len(restaurant), p=[p, 1-p])
restaurant_nan = restaurant.copy(deep=True)
restaurant_nan['total_bill'] = restaurant_nan['total_bill'] * mask
print("Missing bills:", int(restaurant_nan['total_bill'].isna().sum()))
print(restaurant_nan.head(10).to_string(index=False))


## Example 12


In [ ]:
after_counts = restaurant_nan.groupby('time')
print(after_counts['total_bill'].count().to_string())


## Example 13


In [ ]:
missing_trans = lambda x: x.fillna(x.mean())
restaurant_imputed = restaurant_nan.copy(deep=True)
start_transform = time.perf_counter()
restaurant_nan_grouped = restaurant_nan.groupby('time')['total_bill']
restaurant_imputed['total_bill'] = restaurant_nan_grouped.transform(missing_trans)
imputation_transform_time = time.perf_counter() - start_transform
print("Time using grouped transform:", imputation_transform_time, "sec")
print("Missing bills after imputation:", int(restaurant_imputed['total_bill'].isna().sum()))
print(restaurant_imputed[['time', 'total_bill']].head(10).to_string())


## Example 14


In [ ]:
restaurant_loop_imputed = restaurant_nan.copy(deep=True)
start_time = time.perf_counter()
mean_din = restaurant_nan.loc[restaurant_nan['time'] == 'Dinner', 'total_bill'].mean()
mean_lun = restaurant_nan.loc[restaurant_nan['time'] == 'Lunch', 'total_bill'].mean()
for index in restaurant_nan.index:
    if pd.isna(restaurant_nan.loc[index, 'total_bill']):
        if restaurant_nan.loc[index, 'time'] == 'Dinner':
            restaurant_loop_imputed.loc[index, 'total_bill'] = mean_din
        else:
            restaurant_loop_imputed.loc[index, 'total_bill'] = mean_lun
imputation_loop_time = time.perf_counter() - start_time
print("Results from the above operation calculated in %s seconds" % imputation_loop_time)
print("Missing bills after imputation:", int(restaurant_loop_imputed['total_bill'].isna().sum()))
print(restaurant_loop_imputed[['time', 'total_bill']].head(10).to_string())


## Example 15


In [ ]:
restaurant_grouped = restaurant.groupby('day')
filter_trans = lambda x: x['total_bill'].mean() > 20
restaurant_filtered = restaurant_grouped.filter(filter_trans)
print("Retained days:", sorted(restaurant_filtered['day'].unique().tolist()))
print("Retained rows:", len(restaurant_filtered))
print(restaurant_filtered.head().to_string(index=False))


## Example 16


In [ ]:
print(restaurant_filtered['tip'].mean())


## Example 17


In [ ]:
print(restaurant['tip'].mean())


## Example 18


In [ ]:
t = [restaurant.loc[restaurant['day'] == day, 'tip']
     for day in restaurant['day'].unique()
     if restaurant.loc[restaurant['day'] == day, 'total_bill'].mean() > 20]
manual_filtered_tips = pd.concat(t, ignore_index=True) if t else pd.Series(dtype='float64')
print("Retained rows:", len(manual_filtered_tips))
print("Mean tip:", manual_filtered_tips.mean())


## Regression checks


In [ ]:
from pandas.testing import assert_frame_equal, assert_series_equal

assert version('pandas') == '3.1.0'
assert restaurant.shape == (244, 7)
assert restaurant.isna().sum().sum() == 0
assert restaurant.groupby('smoker').size().to_dict() == {'No': 151, 'Yes': 93}
counts = restaurant.groupby('smoker').count()
assert counts.loc['No'].eq(151).all() and counts.loc['Yes'].eq(93).all()
assert squares_comprehension == squares_loop
assert len(squares_loop) == 1000000 and squares_loop[-1] == 999998000001
assert bf_result == formula_result == 500000500000
for n in [0, 1, 2, 99]:assert sum_brute_force(n) == sum_formula(n)
assert isinstance(sum_formula(10**20), int)
assert sum_formula(10**20) == (10**20)*(10**20+1)//2
assert restaurant_transformed.columns.tolist() == ['total_bill', 'tip', 'size']
assert restaurant_transformed.index.equals(restaurant.index)
np.testing.assert_allclose(restaurant_transformed.groupby(restaurant['time']).mean(), 0, atol=1e-12)
np.testing.assert_allclose(restaurant_transformed.groupby(restaurant['time']).std(ddof=1), 1, atol=1e-12)
np.testing.assert_allclose(sex_bill_zscore, manual_zscore['total_bill'])
np.testing.assert_allclose(sex_bill_zscore.groupby(restaurant['sex']).mean(), 0, atol=1e-12)
np.testing.assert_allclose(sex_bill_zscore.groupby(restaurant['sex']).std(ddof=1), 1, atol=1e-12)
assert_frame_equal(manual_zscore.drop(columns='total_bill'), restaurant.drop(columns='total_bill'))

assert restaurant.groupby('time')['total_bill'].count().to_dict() == {'Dinner': 176, 'Lunch': 68}
assert restaurant_nan.groupby('time')['total_bill'].count().to_dict() == {'Dinner': 164, 'Lunch': 59}
missing_bills = restaurant_nan['total_bill'].isna()
assert missing_bills.sum() == 21 and (~missing_bills).sum() == 223
assert_frame_equal(restaurant_imputed, restaurant_loop_imputed)
assert restaurant_imputed['total_bill'].notna().all()
np.testing.assert_array_equal(restaurant_imputed.loc[~missing_bills, 'total_bill'], restaurant.loc[~missing_bills, 'total_bill'])
group_means = restaurant_nan.groupby('time')['total_bill'].transform('mean')
np.testing.assert_allclose(restaurant_imputed.loc[missing_bills, 'total_bill'], group_means.loc[missing_bills])
np.testing.assert_allclose(restaurant_nan.groupby('time')['total_bill'].mean(), restaurant_imputed.groupby('time')['total_bill'].mean())
assert 'total_time' not in restaurant_loop_imputed.columns
assert_frame_equal(restaurant_imputed.drop(columns='total_bill'), restaurant.drop(columns='total_bill'))
assert_frame_equal(restaurant_nan.drop(columns='total_bill'), restaurant.drop(columns='total_bill'))

assert sorted(restaurant_filtered['day'].unique().tolist()) == ['Sat', 'Sun']
assert len(restaurant_filtered) == len(manual_filtered_tips) == 163
assert restaurant_filtered['day'].value_counts().to_dict() == {'Sat': 87, 'Sun': 76}
np.testing.assert_allclose(sorted(manual_filtered_tips), sorted(restaurant_filtered['tip']))
np.testing.assert_allclose(manual_filtered_tips.mean(), restaurant_filtered['tip'].mean())
np.testing.assert_allclose(restaurant_filtered['tip'].mean(), 3.115276073619632)
np.testing.assert_allclose(restaurant['tip'].mean(), 2.99827868852459)
assert (restaurant_filtered['total_bill'] < 20).any()
assert_frame_equal(restaurant_filtered, restaurant.loc[restaurant['day'].isin(['Sat', 'Sun'])])
assert_frame_equal(restaurant, restaurant_original)

# Test all-missing and singleton groups without pretending they have a known mean.
edge = pd.DataFrame({'time': ['A', 'A', 'B', 'B', 'C'],
                     'total_bill': [np.nan, np.nan, 10.0, np.nan, 8.0]})
edge_result = edge.groupby('time')['total_bill'].transform(missing_trans)
assert edge_result.iloc[:2].isna().all()
assert edge_result.iloc[2:].tolist() == [10.0, 10.0, 8.0]
assert edge['total_bill'].isna().sum() == 3
threshold = pd.DataFrame({'day': ['A', 'A', 'B', 'B'], 'total_bill': [20.0, 20.0, 21.0, 23.0]})
assert threshold.groupby('day').filter(filter_trans)['day'].tolist() == ['B', 'B']
assert threshold.groupby('day').filter(lambda group: group['total_bill'].mean() > 100).empty
empty_t = []
empty_tips = pd.concat(empty_t, ignore_index=True) if empty_t else pd.Series(dtype='float64')
assert empty_tips.empty
print('All full-dataset grouping, z-score, missing-value, filtering, exact-integer and input-preservation checks passed.')
print('Restaurant records:', len(restaurant))
print('Seeded missing bills:', int(missing_bills.sum()), 'remaining after imputation:', int(restaurant_imputed['total_bill'].isna().sum()))
print('Retained days:', sorted(restaurant_filtered['day'].unique().tolist()), 'rows:', len(restaurant_filtered))
print('Mean tips: filtered =', restaurant_filtered['tip'].mean(), '; overall =', restaurant['tip'].mean())
